# Praktikum 8 - Jaringan Syaraf Tiruan 3
## Convolutional Neural Network (CNN) untuk Klasifikasi Gambar
**Dataset:** Rock Paper Scissors

## Langkah 3: Import Library

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, Conv2D, MaxPooling2D
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import matplotlib.pyplot as plt

print('TensorFlow version:', tf.__version__)

## Langkah 4: Persiapan Data menggunakan ImageDataGenerator

ImageDataGenerator digunakan untuk mempersiapkan data latih dan data validasi. Fungsi ini menyediakan kemudahan seperti preprocessing data, pelabelan sampel otomatis, dan augmentasi gambar.

In [ ]:
dataset_path = "./rockpaperscissors"

train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

train_generator = train_datagen.flow_from_directory(
    dataset_path,
    target_size=(150, 150),
    batch_size=32,
    class_mode='categorical',
    subset='training',
)

validation_generator = train_datagen.flow_from_directory(
    dataset_path,
    target_size=(150, 150),
    batch_size=32,
    class_mode='categorical',
    subset='validation',
)

print('Kelas yang terdeteksi:', train_generator.class_indices)
print(f'Jumlah data training  : {train_generator.samples}')
print(f'Jumlah data validasi  : {validation_generator.samples}')

## Langkah 5: Membangun Arsitektur Model CNN

Model CNN mirip dengan MLP namun dengan penambahan layer konvolusi dan max pooling. Layer konvolusi mengekstraksi fitur penting dari gambar, sementara max pooling mengurangi resolusi gambar untuk mempercepat proses pelatihan.

In [ ]:
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(150, 150, 3)),
    MaxPooling2D(2, 2),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),
    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),
    Flatten(),
    Dense(512, activation='relu'),
    Dense(3, activation='softmax')
])

model.summary()

## Langkah 6: Kompilasi Model

Kompilasi model menggunakan loss function `categorical_crossentropy` untuk klasifikasi multi-kelas dan optimizer `Adam`.

In [ ]:
model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

print('Model berhasil dikompilasi!')

## Langkah 7: Pelatihan Model (Model Fitting)

Model dilatih menggunakan data latih yang sudah disiapkan. ImageDataGenerator secara otomatis memberikan label sesuai dengan direktori tempat gambar berada.

In [ ]:
history = model.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=10
)

## Langkah 8: Evaluasi Model

In [ ]:
val_loss, val_acc = model.evaluate(validation_generator)
print(f'Validation loss: {val_loss}, Validation accuracy: {val_acc}')

## Langkah 9: Prediksi pada Data Validasi

In [ ]:
predictions = model.predict(validation_generator)
print(predictions)  # Output berupa probabilitas prediksi tiap kelas

## Visualisasi Akurasi dan Loss Training

In [ ]:
plt.figure(figsize=(12, 4))

# Plot Accuracy
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

# Plot Loss
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.savefig('training_history.png')
plt.show()
print('Plot disimpan sebagai training_history.png')